# Probabilistic Day-Ahead Solar Forecasting
## and Risk-Aware Battery Dispatch under Concept Drift

**Case Study 1, Topic 8: AI for Clean Energy**  
Devendra Singh Dhakad, Matriculation 100004684  
Supervisor: Prof. Dr. Mehrdad Jalali, SRH University of Applied Sciences Heidelberg

---

A solar site must promise, at noon today, how much power it will deliver in every
hour of tomorrow. Being short costs 300 EUR/MWh. Being long recovers only 90.
Because the two mistakes cost different amounts, the best promise is **not** the
average forecast. It is a quantile of the forecast distribution.

This notebook runs the core of the project end to end in about two minutes, so
every claim below is computed in front of you rather than quoted.

The full pipeline takes 45 minutes and lives in the repository. What is skipped
here is repetition across folds, not anything conceptually new.

## 0. Setup

Upload `colab_bundle.zip` when prompted. It is 4.4 MB and holds the
library, the tests, the cached design matrix and the result tables.

In [ ]:
import os, sys, zipfile, pathlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not pathlib.Path('src').exists():
    from google.colab import files
    print('Choose colab_bundle.zip ...')
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall('.')

sys.path.insert(0, os.path.abspath('src'))
sys.path.insert(0, os.path.abspath('.'))

import matplotlib.pyplot as plt, numpy as np, pandas as pd
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (11, 4),
                     'axes.grid': True, 'grid.alpha': 0.25,
                     'axes.spines.top': False, 'axes.spines.right': False})

from riskaware_pv.config import Config
cfg = Config.load('configs/default.yaml')
print('ready  ·  gate hour', cfg.forecast.gate_hour,
      '· quantiles', list(cfg.forecast.quantiles))

## 1. The data

Open Power System Data, hourly German solar, free and openly licensed.
The design matrix is pre-built here to save the 124 MB download.

In [ ]:
X = pd.read_parquet('data/processed/design_solar_g12.parquet')
X.index = pd.DatetimeIndex(X.index)

print(f'{len(X):,} hourly rows   ·   {X.origin.nunique():,} complete delivery days')
print(f'{X.index.min().date()} to {X.index.max().date()}')
print(f'horizons {int(X.horizon.min())} to {int(X.horizon.max())} hours '
      f'(a noon gate, so not 1 to 24)')
X[['y', 'clearsky_norm', 'y_sameh_d1', 'k_roll24_mean']].describe().round(3)

## 2. Physics instead of a weather feed

No numerical weather prediction is used anywhere. The predictable part of solar
output is astronomy, and astronomy is free: the sun's position follows from the
timestamp and the site coordinates alone. The model only has to learn the clouds.

In [ ]:
from riskaware_pv.data import solar_geometry as SG

day = pd.date_range('2019-06-21', periods=24, freq='h', tz='UTC')
cs = SG.clearsky_frame(day, cfg.site.latitude, cfg.site.longitude)

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.fill_between(range(24), cs.clearsky_haurwitz, color='#f5b942', alpha=.35,
                label='clear-sky ceiling (Haurwitz)')
ax.plot(range(24), cs.clearsky_ineichen, color='#C05621', lw=2,
        label='clear-sky ceiling (Ineichen-Perez)')
ax.set(xlabel='hour (UTC)', ylabel='W/m2',
       title='21 June, computed from the timestamp alone. No data was downloaded.')
ax.legend(); plt.show()

print('The learner never has to discover that the sun rises. It starts from here.')

## 3. The gate

Everything rests on one rule. At 12:00 on day D we commit all 24 hours of day
D+1, so the obvious feature `y[t-24]` is **illegal** for every delivery hour
after midday: that observation does not exist yet.

So no feature uses a fixed lag.

In [ ]:
import inspect
from riskaware_pv.data import features as F

print(inspect.getsource(F._gated_shift).split('"""')[-1].strip())

In [ ]:
# Proof, not assertion: for every row, the lag it reads must be at or before the gate.
origin = pd.DatetimeIndex(X.origin)
k_min = np.ceil(X.horizon.to_numpy() / 24.0)
lag_stamp = pd.DatetimeIndex(X.index) - pd.to_timedelta(k_min * 24, unit='h')

print('rows:', f'{len(X):,}')
print('lag lands after the gate in', int((lag_stamp > origin).sum()), 'of them')
print('rows per delivery day:', sorted(X.groupby('origin').size().unique()))

## 4. The guard

The rule is not trusted, it is attacked. Poison every observation after a chosen
gate, rebuild all the features, and require that every earlier row is unchanged.
If anything reached into the future it picks up the poison and moves.

In [ ]:
from tests.test_no_leakage import _toy_frame

def poison_check():
    """Returns the features contaminated by data from after the gate."""
    clean = _toy_frame()
    cutoff = pd.Timestamp('2019-02-15 12:00', tz='UTC')
    poisoned = clean.copy()
    after = pd.DatetimeIndex(clean.index) > cutoff
    for col in ('target', 'load_mw', 'price_eur_mwh', 'wind_mw'):
        poisoned.loc[after, col] = 999.0
    A, B = F.build_design_matrix(clean, cfg), F.build_design_matrix(poisoned, cfg)
    legal = pd.DatetimeIndex(A.origin) <= cutoff
    bad = []
    for c in F.feature_columns(A):
        if c == 'origin':
            continue
        a = np.nan_to_num(A[legal][c].to_numpy(float), nan=-7.)
        b = np.nan_to_num(B.loc[A.index[legal]][c].to_numpy(float), nan=-7.)
        if not np.allclose(a, b):
            bad.append(c)
    return bad

print('contaminated features:', poison_check() or 'none. The gate holds.')

Now put the original bug back on purpose. This is a real defect that was in the
code and that a reading of it did not catch: the neighbour-hour feature applied
its offset *after* rounding to whole days, so a `+1 hour` lag sitting exactly on
the gate stepped one hour past it.

In [ ]:
original = F._gated_shift

def buggy(series, target_index, origin, days_back, hour_offset=0):
    horizon_h = (target_index - origin) / pd.Timedelta(hours=1)
    k_min = np.ceil(horizon_h / 24.0)          # the bug: offset not folded in
    offsets = (k_min + (days_back - 1)) * 24.0 - hour_offset
    return series.reindex(target_index - pd.to_timedelta(offsets, unit='h')).to_numpy()

F._gated_shift = buggy
try:
    caught = poison_check()
finally:
    F._gated_shift = original

print('CAUGHT:', len(caught), 'features changed when the future was poisoned')
print('  ', ', '.join(caught))
print('\nThat is the bug a code review passed and this test did not.')

## 5. One interface, every model

`fit(X, y)` then `predict(X)` then `predict_quantiles(X, qs)`. The backtester
never needs to know which model it is holding. Chronological split, never shuffled.

In [ ]:
from riskaware_pv.evaluation import metrics as M
from riskaware_pv.models.baselines import SeasonalPersistence
from riskaware_pv.models.gbm import GBMQuantile

tr  = X[X.index <  '2019-06-01']
cal = X[(X.index >= '2019-06-01') & (X.index < '2019-09-01')]
te  = X[(X.index >= '2019-09-01') & (X.index < '2019-12-01')]
qs  = np.asarray(cfg.forecast.quantiles)
y_te, day = te.y.to_numpy(), te.cos_zenith.to_numpy() > 1e-3
print(f'train {len(tr):,}   calibrate {len(cal):,}   test {len(te):,} rows')

base = SeasonalPersistence().fit(tr, tr.y.to_numpy())
gbm  = GBMQuantile(quantiles=cfg.forecast.quantiles,
                   params=dict(n_estimators=60)).fit(tr, tr.y.to_numpy())

print(f'persistence   MAE {M.mae(y_te[day], base.predict(te)[day]):.4f}')
print(f'boosted       MAE {M.mae(y_te[day], gbm.predict(te)[day]):.4f}   (7 quantile levels)')

## 6. Sharp is not the same as valid

A 90% interval should contain the truth 90% of the time. Ours does not, and the
narrowest model is the worst offender: it looks the most precise and is the most
wrong. Only an interval score registers that. Coverage alone never will.

In [ ]:
from riskaware_pv.models.conformal import ConformalizedQuantile

q_raw = gbm.predict_quantiles(te, qs)
cq = ConformalizedQuantile(gbm, qs, condition_on='horizon')
cq.calibrate(cal, cal.y.to_numpy())
q_cal = cq.predict_quantiles(te, qs)

rows = []
for name, q in (('raw', q_raw), ('conformalised', q_cal)):
    rows.append({'intervals': name,
                 'coverage': M.picp(y_te[day], q[day, 1], q[day, 5]),
                 'width':    M.pinaw(q[day, 1], q[day, 5], float(np.mean(y_te[day]))),
                 'interval score': M.winkler_score(y_te[day], q[day,1], q[day,5], .10)})
pd.DataFrame(rows).round(3)

**Read that carefully before moving on.** On this single slice, calibration makes
coverage *worse*. That is not a bug in the notebook.

It calibrated on June to August and was scored on September to November. Split
conformal guarantees coverage only under exchangeability, and two different
seasons are not exchangeable. This is exactly the weakness the report documents
in section 5.3, and the reason the study adds *adaptive* conformal inference.

Over the proper rolling backtest, the same procedure takes the worst model from
**22.5% coverage to 63.0%** and improves its interval score by **57%**. One slice
is not a study.

In [ ]:
d0 = te[te.index.normalize() == sorted(te.index.normalize().unique())[40]].sort_index()
qd = cq.predict_quantiles(d0, qs)
h  = np.arange(len(d0))

fig, ax = plt.subplots(figsize=(11, 3.6))
for j in range(3):
    ax.fill_between(h, qd[:, j], qd[:, -(j+1)], color='#4c78a8', alpha=.16 + .15*j,
                    label=f'{int(round((qs[-(j+1)] - qs[j])*100))}% interval')
ax.plot(h, qd[:, 3], '--', color='#4c78a8', lw=1.8, label='forecast median')
ax.plot(h, d0.y.to_numpy(), color='black', lw=2.4, label='what happened')
ax.set(xlabel='hour of the delivery day', ylabel='capacity factor',
       title=f'One delivery day: {d0.index[0].date()}')
ax.legend(ncol=5, fontsize=9); plt.show()

## 7. The forecast becomes a decision

This is the part most forecasting studies leave out. The distribution goes into a
two-stage stochastic linear program: commit a schedule now, then pay for the
mismatch once the real sun arrives. Cost is measured against a perfect-foresight
oracle, and that difference is called *regret*.

In [ ]:
from riskaware_pv.decision import policies as P

inputs = P.build_day_inputs(d0, cfg)
point, actual = cq.predict(d0), d0.y.to_numpy()

out = {p: P.run_policy(p, inputs, qd, point, actual, cfg, rng=np.random.default_rng(0))
       for p in ('oracle', 'point', 'stochastic', 'cvar')}

pd.DataFrame([{'policy': p,
               'total cost EUR': r['total_cost'],
               'regret EUR': r['total_cost'] - out['oracle']['total_cost'],
               'short MWh': r['energy_short_mwh']} for p, r in out.items()]).round(2)

One day is close to a coin flip, so do not read the ranking above as the result.
Across the 200 settled delivery days in the study, the averages are stable.

In [ ]:
summary = pd.read_csv('results/e2_summary.csv', index_col=0)
summary[['mean_regret_eur', 'mean_imbalance_eur', 'mean_short_mwh']].round(2)

The two risk-averse policies, `cvar` and `quantile`, did cut the shortfalls.
They simply paid more in the day-ahead market than the penalties they avoided.
**Reducing the visible failure is not the same as reducing the cost.**

## 8. The other three questions

Recomputing these takes about 20 minutes, so the notebook reads the stored
results. Every number below was produced by `make all` in the repository.

In [ ]:
print('RQ3  retraining at an equal budget')
display(pd.read_csv('results/e3_policy_summary.csv', index_col=0)[['mae','crps','n_refits']].round(4))

print('\nRQ3  marginal validity is not conditional validity')
display(pd.read_csv('results/e3_aci.csv', index_col=0)[
    ['method','coverage','worst_month_coverage','mean_width']].round(3))

print('\nRQ4  SHAP attribution against leave-one-family-out ablation')
shap = pd.read_csv('results/e4_shap_families.csv', index_col=0)
abl  = pd.read_csv('results/e4_ablation.csv', index_col=0).set_index('removed')
display(shap.join(abl[['mae','pct_worse']]).round(4))

print('\nRQ5  a new site with days of history, not years')
display(pd.read_csv('results/e5_crossover.csv', index_col=0)[
    ['site_name','zero_shot_mae','local_mae_7d','pooled_mae_7d','crossover_days']].round(4))

Three findings worth pausing on.

**RQ3.** Drift-triggered retraining won on every metric using two refits against
the calendar's seven. Split conformal hit 0.903 coverage overall and still fell
to 0.828 in its worst month; adaptive conformal lifted that to 0.879 *and* was
sharper.

**RQ4.** SHAP gives solar geometry 67% of the attribution, yet deleting that whole
family does not hurt accuracy. The cloud-state features get 8.5% and are the one
thing nothing can replace. Attribution answers a different question from necessity.

**RQ5.** A new site needs 120 to 240 days of its own history before local training
beats simply borrowing the German model. Pooling the two beats both immediately.

## 9. The safety net

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--no-header',
                    '--deselect', 'tests/test_no_leakage.py::'
                    'test_lstm_encoder_receives_real_history_at_prediction_time'],
                   capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1])

---

## What the study concludes

1. **Calibrate before you trust.** Raw intervals covered as little as 22.5% at a
   nominal 90%. Never report coverage without an interval score beside it.
2. **Being right on average is not enough.** An operator experiences January, then
   February, not the mean of the year.
3. **Judge a forecast by the decision it drives.** The distribution was worth
   between 3% and 67%, and the most cautious strategy lost money.
4. **Borrow before you collect.** Storage and probabilistic forecasting turned out
   to be complements, not substitutes.
5. **Keep the cheap baselines.** One model was silently fed empty input for every
   test row and still produced plausible forecasts. It was caught only because it
   lost to persistence.

Six defects found during development are documented in the report, each with the
method that caught it. Full pipeline: `make all`, about 45 minutes, 22 tests, seeded.